In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-addon-utils
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-addon-utils~=0.3.0
```
</details>

حزمة أدوات Qiskit addon المساعدة هي مجموعة من الوظائف التي تُكمّل سير العمل التي تتضمن addon واحدة أو أكثر من Qiskit addons. على سبيل المثال، تحتوي هذه الحزمة على دوال لإنشاء الهاملتونيات (Hamiltonians)، وتوليد دوائر التطور الزمني (Trotter time-evolution circuits)، وتقسيم الدوائر الكمومية ودمجها.

## التثبيت

يوجد طريقتان لتثبيت حزمة أدوات Qiskit addon المساعدة: عبر PyPI أو البناء من المصدر. يُنصح بتثبيت هذه الحزم داخل [بيئة افتراضية](https://docs.python.org/3.10/tutorial/venv.html) لضمان عزل تبعيات الحزم.

### التثبيت من PyPI

أسهل طريقة لتثبيت حزمة أدوات Qiskit addon المساعدة هي عبر PyPI.

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit import QuantumCircuit
from qiskit_addon_utils.coloring import auto_color_edges
from qiskit_addon_utils.slicing import combine_slices, slice_by_depth
from collections import defaultdict

backend = FakeSherbrooke()
coupling_map = backend.coupling_map

# Create naive circuit
circuit = QuantumCircuit(backend.num_qubits)
for edge in coupling_map.graph.edge_list():
    circuit.cz(edge[0], edge[1])


# Color the edges of the coupling map
coloring = auto_color_edges(coupling_map)
circuit_with_coloring = QuantumCircuit(backend.num_qubits)

# Make a reverse coloring dict in order to make the circuit
color_to_edge = defaultdict(list)
for edge, color in coloring.items():
    color_to_edge[color].append(edge)

# Place edges in order of color
for edges in color_to_edge.values():
    for edge in edges:
        circuit_with_coloring.cz(edge[0], edge[1])

print(f"The circuit without using edge coloring has depth: {circuit.depth()}")
print(
    f"The circuit using edge coloring has depth: {circuit_with_coloring.depth()}"
)

The circuit without using edge coloring has depth: 37
The circuit using edge coloring has depth: 3


### التثبيت من المصدر
<details>
<summary>
Click here to read how to install this package manually.
</summary>

إذا أردت المساهمة في هذه الحزمة أو تثبيتها يدويًا، ابدأ باستنساخ المستودع:

In [2]:
import numpy as np
from qiskit import QuantumCircuit

num_qubits = 9
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
qubits_1 = [i for i in range(num_qubits) if i % 2 == 0]
qubits_2 = [i for i in range(num_qubits) if i % 2 == 1]
qc.cx(qubits_1[:-1], qubits_2)
qc.cx(qubits_2, qubits_1[1:])
qc.cx(qubits_1[-1], qubits_1[0])
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))
qc.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/d5a5c53f-45d2-4cdc-86f3-221f179cdbea-0.svg" alt="Output of the previous code cell" />

ثم ثبّت الحزمة عبر `pip`. إذا كنت تخطط لتشغيل الدروس التعليمية الموجودة في مستودع الحزمة، ثبّت تبعيات notebook أيضًا. وإذا كنت تخطط للتطوير في المستودع، ثبّت تبعيات `dev`.

In [3]:
# Slice circuit into partitions of depth 1
slices = slice_by_depth(qc, 1)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/6d824d97-edc6-4c88-bcfa-428731393f83-0.svg" alt="Output of the previous code cell" />

</details>

## البدء باستخدام الأدوات المساعدة
توجد عدة وحدات (modules) داخل حزمة `qiskit-addon-utils`، منها وحدة لتوليد المسائل لمحاكاة الأنظمة الكمومية، وأخرى لتلوين الرسم البياني (graph coloring) لوضع البوابات في الدائرة الكمومية بصورة أكثر كفاءة، ووحدة لتقسيم الدوائر (circuit slicing) التي يمكن أن تساعد في [الانتشار العكسي للمؤثرات (operator backpropagation)](/guides/qiskit-addons-obp). تُلخّص الأقسام التالية كل وحدة. كما تحتوي [وثائق API](https://docs.quantum.ibm.com/api/qiskit-addon-utils) الخاصة بالحزمة على معلومات مفيدة.

### توليد المسائل
محتويات وحدة [`qiskit_addon_utils.problem_generators`](../api/qiskit-addon-utils/problem-generators) تشمل:

- دالة [`generate_xyz_hamiltonian()`](../api/qiskit-addon-utils/problem-generators#generate_xyz_hamiltonian)، التي تولّد تمثيل `SparsePauliOp` الواعي بالاتصالية لنموذج XYZ من نوع إيزينغ (Ising-type):

$$H = \sum_{(j,k)\in E} \left(J_x X_jX_k + J_yY_jY_k + J_zZ_jZ_k\right) + \sum_{j\in V} \left(h_x X_j + h_y Y_j + h_z Z_j\right) $$
- دالة [`generate_time_evolution_circuit()`](../api/qiskit-addon-utils/problem-generators#generate_time_evolution_circuit)، التي تبني دائرة تُمثّل التطور الزمني لمؤثر معيّن.
- ثلاثة كائنات [`PauliOrderStrategy`](../api/qiskit-addon-utils/problem-generators#pauliorderstrategy) مختلفة لتعداد ترتيبات سلاسل باولي (Pauli strings) المتعددة. يكون هذا مفيدًا بصورة خاصة عند استخدامه مع تلوين الرسم البياني، ويمكن استخدامه كوسيطات في دالتي `generate_xyz_hamiltonian()` و`generate_time_evolution_circuit()`.

### تلوين الرسم البياني
تُستخدم وحدة [`qiskit_addon_utils.coloring`](../api/qiskit-addon-utils/coloring) لتلوين حواف خريطة الاقتران (coupling map) واستخدام هذا التلوين لوضع البوابات في الدائرة الكمومية بكفاءة أعلى. الهدف من خريطة الاقتران ذات الحواف الملوّنة هو إيجاد مجموعة من ألوان الحواف بحيث لا يتشارك حافّتان من اللون نفسه في عقدة (node) مشتركة. بالنسبة لوحدة المعالجة الكمومية (QPU)، هذا يعني أن البوابات الموجودة على الحواف ذات اللون نفسه (اتصالات الكيوبتات) يمكن تشغيلها بالتوازي، مما يجعل الدائرة تُنفَّذ بشكل أسرع.

كمثال سريع، يمكنك استخدام دالة [`auto_color_edges()`](../api/qiskit-addon-utils/coloring#auto_color_edges) لتوليد تلوين للحواف لدائرة بسيطة تُنفّذ بوابة `CZGate` على كل اتصال كيوبت. يستخدم مقطع الكود أدناه خريطة اقتران الـ backend الخاصة بـ `FakeSherbrooke`، ثم يبني هذه الدائرة البسيطة، ويستخدم دالة `auto_color_edges()` لإنشاء دائرة مكافئة أكثر كفاءة.

In [4]:
from qiskit_addon_utils.slicing import slice_by_gate_types

slices = slice_by_gate_types(qc)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/d011c56e-d741-491a-8867-54952cb7b757-0.svg" alt="Output of the previous code cell" />

If your workload is designed to exploit the physical qubit connectivity of the QPU it will be run on, you can create slices based on edge coloring. The code snippet below will assign a three-coloring to circuit edges and slice the circuit with respect to the edge coloring. (Note: this only affects non-local gates. Single-qubit gates will be sliced by gate type).

In [5]:
from qiskit_addon_utils.slicing import slice_by_coloring

# Assign a color to each set of connected qubits
coloring = {}
for i in range(num_qubits - 1):
    coloring[(i, i + 1)] = i % 3
coloring[(num_qubits - 1, 0)] = 2

# Create a circuit with operations added in order of color
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
edges = [
    edge for color in range(3) for edge in coloring if coloring[edge] == color
]
for edge in edges:
    qc.cx(edge[0], edge[1])
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))

# Create slices by edge color
slices = slice_by_coloring(qc, coloring=coloring)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/41290d70-e116-4bd2-b9e7-d90aeaa852aa-0.svg" alt="Output of the previous code cell" />

### التقسيم
أخيرًا، تحتوي وحدة [`qiskit-addon-utils.slicing`](../api/qiskit-addon-utils/slicing) على دوال ومسارات transpiler للعمل مع إنشاء "شرائح" الدوائر (circuit slices)، وهي أقسام زمنية لـ [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit) تمتد عبر جميع الكيوبتات. تُستخدم هذه الشرائح بصورة رئيسية في [الانتشار العكسي للمؤثرات (operator backpropagation)](/guides/qiskit-addons-obp-get-started). الطرق الرئيسية الأربع لتقسيم الدائرة هي: حسب نوع البوابة، أو العمق، أو التلوين، أو تعليمات [`Barrier`](../api/qiskit/circuit#barrier). يُعيد ناتج دوال التقسيم قائمة من كائنات [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit). يمكن أيضًا إعادة دمج الشرائح باستخدام دالة [`combine_slices()`](../api/qiskit-addon-utils/slicing#combine_slices). اقرأ [مرجع API](../api/qiskit-addon-utils/slicing) الخاص بالوحدة للمزيد من المعلومات.

فيما يلي بعض الأمثلة على كيفية إنشاء هذه الشرائح باستخدام الدائرة التالية:

In [6]:
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
qc.barrier()
qubits_1 = [i for i in range(num_qubits) if i % 2 == 0]
qubits_2 = [i for i in range(num_qubits) if i % 2 == 1]
qc.cx(qubits_1[:-1], qubits_2)
qc.cx(qubits_2, qubits_1[1:])
qc.cx(qubits_1[-1], qubits_1[0])
qc.barrier()
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))
qc.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/5040a678-9d5f-4c8b-8a92-7de04c3fcfda-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-utils/extracted-outputs/d5a5c53f-45d2-4cdc-86f3-221f179cdbea-0.svg)

في الحالات التي لا توجد فيها طريقة واضحة لاستغلال بنية الدائرة في الانتشار العكسي للمؤثرات، يمكنك تقسيم الدائرة إلى شرائح بعمق محدد.

In [7]:
from qiskit_addon_utils.slicing import slice_by_barriers

slices = slice_by_barriers(qc)
slices[0].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/b1106c2f-711c-4d30-b91a-ea05f47598d8-0.svg" alt="Output of the previous code cell" />

In [8]:
slices[1].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/1afd2111-dd88-4e20-a137-f0c975e9d1bb-0.svg" alt="Output of the previous code cell" />

In [9]:
slices[2].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/360ab773-df81-4975-bb19-cd5f34e69b29-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-utils/extracted-outputs/41290d70-e116-4bd2-b9e7-d90aeaa852aa-0.svg)

إذا كان لديك استراتيجية تقسيم مخصصة، يمكنك بدلاً من ذلك وضع حواجز (barriers) في الدائرة لتحديد نقاط التقسيم، ثم استخدام دالة [`slice_by_barriers`](../api/qiskit-addon-utils/slicing#slice_by_barriers).